In [1]:
import numpy as np
import pandas as pd
import os

folder_path = os.getcwd()
dataset_folder_path_raw = os.path.join(folder_path,"..\\data_raw\\")
dataset_folder_path_clean = os.path.join(folder_path,"..\\data_cleaned\\")

order_items_dataset = os.path.join(dataset_folder_path_clean,"order_items_dataset.csv")
order_items_df = pd.read_csv(order_items_dataset)
order_items_df.columns

Index(['Unnamed: 0', 'order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')

In [2]:
order_payments_dataset = os.path.join(dataset_folder_path_raw,"order_payments_dataset.csv")
order_payments_df = pd.read_csv(order_payments_dataset)
order_payments_df.columns

Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='str')

In [3]:
order_reviews_dataset = os.path.join(dataset_folder_path_raw,"order_reviews_dataset.csv")
order_reviews_df = pd.read_csv(order_reviews_dataset)
order_reviews_df.columns

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='str')

In [4]:
orders_dataset = os.path.join(dataset_folder_path_raw,"orders_dataset.csv")
orders_df = pd.read_csv(orders_dataset)
orders_df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

### Bringing all order related data together

1. Orders
   1. customer_id
   2. order_status
   3. order_purchase_time, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date

2. Order reviews
   1. Review score

3. Order Payments
   1.  payment_type
   2.  payment_installments
   3.  payment_value

In [5]:
# Merging all  with order_items

orders_subset = orders_df[
    [
        'order_id',
        'customer_id',
        'order_status',
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
]

# Order Reviews: keep only required columns
reviews_subset = order_reviews_df[['order_id','review_score']]

# Order Payments: keep only required columns
payments_subset = order_payments_df[
    [
        'order_id',
        'payment_type',
        'payment_installments',
        'payment_value'
    ]
]

# Merge into order_items
order_items_enriched = (
    order_items_df
    .merge(orders_subset, on='order_id', how='left')
    .merge(reviews_subset, on='order_id', how='left')
    .merge(payments_subset, on='order_id', how='left')
)

print(order_items_enriched.shape)
order_items_enriched.columns

(118310, 19)


Index(['Unnamed: 0', 'order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'customer_id',
       'order_status', 'order_purchase_timestamp', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'review_score', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='str')

In [6]:
order_items_enriched = order_items_enriched.drop(columns=['Unnamed: 0'], errors='ignore')

In [7]:
order_items_enriched.duplicated().sum()

np.int64(1104)

In [8]:
payments_agg = (
    order_payments_df
    .groupby('order_id', as_index=False)
    .agg({
        'payment_type': lambda x: ', '.join(x.astype(str).unique()),
        'payment_installments': 'sum',
        'payment_value': 'sum'
    })
)

order_items_enriched1 = (
    order_items_df
    .merge(orders_subset, on='order_id', how='left')
    .merge(reviews_subset, on='order_id', how='left')
    .merge(payments_agg, on='order_id', how='left')
)

order_items_enriched1.duplicated().sum()

np.int64(399)

In [9]:
order_items_enriched1.columns

Index(['Unnamed: 0', 'order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'customer_id',
       'order_status', 'order_purchase_timestamp', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'review_score', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='str')

In [10]:
order_items_enriched1 = order_items_enriched1.drop(columns=['Unnamed: 0'], errors='ignore')

In [11]:
order_items_enriched1.duplicated().sum()

np.int64(399)

In [12]:
dup_perc = order_items_enriched1.duplicated().sum() / order_items_enriched1.shape[0]
dup_perc

np.float64(0.0035211889086961892)

In [13]:
order_items_enriched1.drop_duplicates(inplace=True)

In [14]:
order_items_enriched1.duplicated().sum()

np.int64(0)

In [ ]:
output_file_path = os.path.join(folder_path,"..\\data_model\\facts_order_items.csv")
order_items_enriched1.to_csv(output_file_path)